# MLE — Train Arcade Game RL Agent on Colab GPU

Trains Dreamer v3 on any MAME arcade game using GPU acceleration.

**Upload your ROM zip** (e.g. `qbert.zip`) when prompted.

In [ ]:
# 1. Install MAME + dependencies
!apt-get update -qq && apt-get install -y -qq mame xvfb > /dev/null 2>&1
!apt-get install -y -qq python3.11 python3.11-venv python3.11-dev > /dev/null 2>&1

# Start virtual display for SDL
import subprocess, os, shutil
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1x1x24'])
os.environ['DISPLAY'] = ':99'
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'
os.environ['XDG_RUNTIME_DIR'] = '/tmp'
if '/usr/games' not in os.environ.get('PATH', ''):
    os.environ['PATH'] = '/usr/games:' + os.environ['PATH']

# Clone MLE repo
!rm -rf /content/mle && git clone https://github.com/lilwg/mle.git /content/mle
%cd /content/mle

# Install Python 3.11 venv with sheeprl + CUDA torch
!python3.11 -m venv /content/venv311
!source /content/venv311/bin/activate && \
    pip install -q 'numpy<2' && \
    pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121 && \
    pip install -q sheeprl gymnasium pillow wandb opencv-python-headless && \
    pip install -q 'numpy<2' && \
    python -c "import numpy; print(f'numpy={numpy.__version__}'); import cv2; print('cv2 OK')"

print('Setup complete!')

In [ ]:
# 2. Upload ROM file
from google.colab import files
import os

uploaded = files.upload()
os.makedirs('roms', exist_ok=True)
for fname in uploaded:
    os.rename(fname, f'roms/{fname}')
    print(f'ROM: roms/{fname}')

rom_name = [f[:-4] for f in os.listdir('roms') if f.endswith('.zip')][0]
print(f'Game: {rom_name}')

# Verify ROM compatibility
!mame -rompath roms {rom_name} -verifyroms 2>&1

In [ ]:
# 3. Train Dreamer v3 with GPU
GAME = rom_name
TIMESTEPS = 500_000
os.environ['MLE_ROMS_PATH'] = '/content/mle/roms'

print(f'Training Dreamer v3 on {GAME} for {TIMESTEPS} steps...')
!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo 'No GPU'

!source /content/venv311/bin/activate && cd /content/mle && \
    DISPLAY=:99 SDL_VIDEODRIVER=dummy SDL_AUDIODRIVER=dummy \
    MLE_ROMS_PATH=/content/mle/roms \
    python -u dreamer_train.py {GAME} --timesteps {TIMESTEPS}